In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv
from ls_prior import builder as ls_prior_builder

from cardiac_electrophysiology.components import prior
from cardiac_electrophysiology.utils import analysis, mesh_utils, visualization

In [2]:
save_data = False
mesh_path = Path("../data/mesh.vtu")
pv_mesh = pv.read(mesh_path)
dlx_mesh = mesh_utils.create_dolfinx_mesh_from_pyvista_mesh(pv_mesh)

In [3]:
prior_settings = ls_prior_builder.BilaplacianPriorSettings(
    mesh=dlx_mesh,
    mean_vector=np.zeros(dlx_mesh.geometry.x.shape[0]),
    kappa=5,
    tau=0.01,
    seed=0,
)
prior_component = prior.AngleFieldPrior(prior_settings)
sample = prior_component.generate_sample()
sample = analysis.shift_angles_to_minimize_axial_variance(sample).flatten()
mean, variance = analysis.compute_axial_mean_and_variance(sample)
mean = mean * np.ones_like(sample)

In [4]:
if save_data:
    np.save("../data/ground_truth_from_sde.npy", sample)
    np.save("../data/prior_mean_from_sde.npy", mean)
visualization.visualize_scalar_field(
    mesh=pv_mesh,
    scalar_field=sample,
    circular=True,
)

Widget(value='<iframe src="http://localhost:42893/index.html?ui=P_0x7f6757906900_0&reconnect=auto" class="pyvi…